# Manual Tracing: Add Custom Spans to Any Application

Instrument any Python application with custom spans, user context, and metadata — and see every call visualized in the FutureAGI Tracing dashboard.

By the end of this notebook you will have a fully instrumented application that traces OpenAI calls automatically, adds custom spans for non-LLM steps, attaches user and session context, and displays the full execution tree in FutureAGI Tracing.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+
- OpenAI API key

## Install

In [ ]:
%pip install fi-instrumentation-otel traceAI-openai openai --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"          # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"    # Replace with your key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

## Step 1: Auto-trace OpenAI calls in 4 lines

`register()` sets up an OpenTelemetry tracer provider connected to FutureAGI. `OpenAIInstrumentor` patches the OpenAI client so every API call is automatically captured — model, messages, token counts, latency — with no further code changes.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

# 1. Register the tracer provider
trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="my-app",
)

# 2. Patch the OpenAI client
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

# All subsequent OpenAI calls are now traced automatically
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)
print(response.choices[0].message.content)

Go to [app.futureagi.com](https://app.futureagi.com) → **Tracing** (left sidebar) and you will see the call appear with its full input/output and token usage.

## Step 2: Add a custom span for non-LLM steps

Not every meaningful step calls an LLM. Database lookups, retrieval, validation, and preprocessing are invisible to auto-instrumentation. Wrap them in a custom span to include them in your trace tree.

In [ ]:
from fi_instrumentation import FITracer

# Get a tracer scoped to this module
tracer = FITracer(trace_provider.get_tracer(__name__))

def retrieve_context(query: str) -> list[str]:
    with tracer.start_as_current_span("retrieve-context") as span:
        span.set_attribute("retrieval.query", query)

        # Simulate a vector DB lookup
        docs = ["Paris is the capital of France.", "France is in Western Europe."]

        span.set_attribute("retrieval.doc_count", len(docs))
        return docs


def answer_with_context(query: str) -> str:
    # Parent span groups retrieval + LLM into one trace
    with tracer.start_as_current_span("answer-with-context") as span:
        docs = retrieve_context(query)
        context = "\n".join(docs)

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Answer using only this context:\n{context}"},
                {"role": "user", "content": query},
            ],
        )
        return response.choices[0].message.content


print(answer_with_context("Where is Paris?"))

In the dashboard, the trace tree shows `answer-with-context` (parent) → `retrieve-context` + OpenAI LLM span (children), with per-step timing.

## Step 3: Attach user ID, session ID, and metadata

Context managers from `fi_instrumentation` propagate attributes to every span created inside them. Any LLM call or custom span inside the `with` block inherits these values automatically.

In [ ]:
from fi_instrumentation import using_user, using_session, using_metadata

user_id = "user-abc123"
session_id = "session-xyz789"
metadata = {"environment": "production", "app_version": "2.1.0"}

with using_user(user_id), using_session(session_id), using_metadata(metadata):
    result = answer_with_context("What is the capital of France?")
    print(result)

In the Tracing dashboard, `userId` is available as a direct filter. To filter by `session.id` or `metadata`, use the **Attribute** filter.

## Step 4: Tag spans for filtering and alerting

Tags are string labels that let you group traces by environment, feature flag, experiment branch, or any other category.

In [ ]:
from fi_instrumentation import using_tags

with using_tags(["production", "rag-pipeline", "v2"]):
    result = answer_with_context("Who wrote Hamlet?")
    print(result)

In Tracing, filter by tags using the **Attribute** filter: select **Attribute** → pick `tag.tags` → set operator to **Contains** → enter `rag-pipeline`.

## Step 5: Nest spans for complex multi-step pipelines

For multi-step operations, nest spans to show the execution hierarchy. A parent span groups related child spans — the total latency of the parent reflects the sum of its children.

In [ ]:
def run_rag_pipeline(user_query: str, user_id: str, session_id: str) -> str:
    with using_user(user_id), using_session(session_id), using_tags(["rag-pipeline"]):
        with tracer.start_as_current_span("rag-pipeline") as pipeline_span:
            pipeline_span.set_attribute("pipeline.query", user_query)

            # Child span 1: retrieval
            with tracer.start_as_current_span("retrieve") as retrieval_span:
                docs = retrieve_context(user_query)
                retrieval_span.set_attribute("retrieval.doc_count", len(docs))

            # Child span 2: LLM call (auto-instrumented)
            context_text = "\n".join(docs)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": f"Answer using:\n{context_text}"},
                    {"role": "user", "content": user_query},
                ],
            )
            answer = response.choices[0].message.content
            pipeline_span.set_attribute("pipeline.answer_length", len(answer))
            return answer


result = run_rag_pipeline(
    user_query="What is the population of France?",
    user_id="user-abc123",
    session_id="session-xyz789",
)
print(result)

The trace tree in Tracing shows: `rag-pipeline` → `retrieve` → OpenAI LLM span, with each step's duration visible.

## Step 6: Log prompt template details with `using_prompt_template`

If you use [prompt versioning](https://docs.futureagi.com/cookbook/quickstart/prompt-versioning), attach the template name, label, and version to every span created inside the block.

In [ ]:
from fi_instrumentation import using_prompt_template

with using_prompt_template(
    template="support-response",
    label="production",
    version="v2",
    variables={"question": "What is the return policy?"},
):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "What is the return policy?"}],
    )
    print(response.choices[0].message.content)

## Step 7: Use decorators for agent and tool spans

`FITracer` provides `@tracer.agent`, `@tracer.chain`, and `@tracer.tool` decorators that automatically capture function inputs and outputs as span attributes.

In [ ]:
@tracer.agent(name="support_agent")
def support_agent(question: str) -> str:
    """Top-level agent that orchestrates retrieval and generation."""
    docs = search_docs(question)
    return generate_answer(question, docs)


@tracer.tool(name="search_docs", description="Search the product documentation")
def search_docs(query: str) -> list[str]:
    return ["30-day return policy for unused items.", "Free shipping on orders over $50."]


@tracer.chain(name="generate_answer")
def generate_answer(question: str, docs: list[str]) -> str:
    context = "\n".join(docs)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer using:\n{context}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


result = support_agent("What is the return policy?")
print(result)

trace_provider.force_flush()

In Tracing, the span tree shows: `support_agent` (agent) → `search_docs` (tool) → `generate_answer` (chain) → OpenAI LLM span. Each decorator sets the `fi.span_kind` attribute (`AGENT`, `TOOL`, or `CHAIN`) so you can filter by span type in the dashboard.

> **Tip:** All decorators support both sync and async functions automatically. They also capture function arguments as `input.value` and the return value as `output.value` on the span.

## What you built

- Auto-traced every OpenAI API call with zero boilerplate using `OpenAIInstrumentor`
- Added custom `retrieve-context` and `rag-pipeline` spans for non-LLM steps, with attributes on each
- Attached `user.id`, `session.id`, and `metadata` to entire request flows using context managers
- Tagged traces with `using_tags` for environment and feature-level filtering
- Nested child spans under a parent to represent a complete RAG pipeline
- Logged prompt template name, label, and version with `using_prompt_template`
- Used `@tracer.agent`, `@tracer.tool`, and `@tracer.chain` decorators for automatic input/output capture

### Next steps

- [Inline Evals in Tracing](https://docs.futureagi.com/cookbook/quickstart/inline-evals-tracing) — attach quality scores to every trace as it runs
- [Session-Based Observability](https://docs.futureagi.com/cookbook/quickstart/session-observability) — wrap multi-turn conversations in user and session context
- [Agent Compass](https://docs.futureagi.com/cookbook/quickstart/agent-compass-debug) — use traced data to automatically surface agent failure patterns
- [Auto-Instrumentation](https://docs.futureagi.com/future-agi/products/observability/auto-instrumentation/overview) — one-line setup for LangChain, LlamaIndex, CrewAI, and 20+ more